# 03 — Feature Engineering

Build the feature matrix from raw data and save to `data/processed/`.

**Input:** `data/raw/playoff_games_*.parquet`, `data/raw/team_metrics_*.parquet`  
**Output:** `data/processed/playoff_features.parquet`

**Feature groups:**
1. Team efficiency — ORtg, DRtg, net rating, pace (10 features)
2. Team quality proxies — win %, offensive rebounding (4 features; replaces player-level data)
3. Game context — rest days, game number, elimination game (5 features)
4. Historical playoff record — prior win %, Finals appearances (4 features)

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path().resolve().parent))
from src.features import add_rest_days_from_playoff_log

In [ ]:
# CONFIG
RAW_DIR = Path().resolve().parent / "data" / "raw"
PROCESSED_DIR = Path().resolve().parent / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = PROCESSED_DIR / "playoff_features.parquet"

SEASONS = [
    "2014-15",
    "2015-16",
    "2016-17",
    "2017-18",
    "2018-19",
    "2020-21",
    "2021-22",
    "2022-23",
    "2023-24",
]

## 1. Load Raw Data

In [ ]:
games = pd.concat(
    [
        pd.read_parquet(RAW_DIR / f"playoff_games_{s}.parquet").assign(season=s)
        for s in SEASONS
    ],
    ignore_index=True,
)
metrics = pd.concat(
    [
        pd.read_parquet(RAW_DIR / f"team_metrics_{s}.parquet").assign(season=s)
        for s in SEASONS
    ],
    ignore_index=True,
)

# Extract playoff round from GAME_ID (chars 6-7: 01=R1, 02=R2, 03=conf finals, 04=Finals)
games["round"] = games["GAME_ID"].str[6:8].astype(int)
games["is_finals"] = (games["round"] == 4).astype(int)
games["GAME_DATE"] = pd.to_datetime(games["GAME_DATE"])

print(f"Games rows: {len(games)}, Metric rows: {len(metrics)}")
print(f"Rounds found: {sorted(games['round'].unique())}")

## 2. One Row Per Game + Rest Days

In [ ]:
# Add rest days computed from playoff game dates
games = add_rest_days_from_playoff_log(games)

# Split home / away
home = games[games["MATCHUP"].str.contains(r"vs\.")].copy()
away = games[games["MATCHUP"].str.contains("@")][
    ["GAME_ID", "TEAM_ID", "rest_days"]
].copy()
away = away.rename(columns={"TEAM_ID": "away_team_id", "rest_days": "away_rest_days"})

# One row per game
df = home.merge(away, on="GAME_ID", how="inner")
df = df.rename(columns={"TEAM_ID": "home_team_id", "rest_days": "home_rest_days"})
df["home_win"] = (df["WL"] == "W").astype(int)
df = df.sort_values(["season", "GAME_DATE"]).reset_index(drop=True)

print(f"Unique games: {len(df)}")
df[["season", "GAME_DATE", "home_team_id", "away_team_id", "home_win", "round"]].head()

## 3. Series Game Number + Elimination Game

In [ ]:
# Sort consistently so the same matchup always has the same 'team_a'/'team_b' key
df["series_key"] = df.apply(
    lambda r: (
        f"{r['season']}_{r['round']}_{min(r['home_team_id'], r['away_team_id'])}_{max(r['home_team_id'], r['away_team_id'])}"
    ),
    axis=1,
)

df = df.sort_values(["series_key", "GAME_DATE"]).reset_index(drop=True)
df["game_number"] = df.groupby("series_key").cumcount() + 1

# Cumulative wins per team in the series up to (not including) this game
df["home_series_wins"] = df.groupby("series_key")["home_win"].cumsum() - df["home_win"]
df["away_series_wins"] = df["game_number"] - 1 - df["home_series_wins"]

df["is_elimination_game"] = (
    (df["home_series_wins"] == 3) | (df["away_series_wins"] == 3)
).astype(int)

print("Game number distribution:")
print(df["game_number"].value_counts().sort_index().to_string())
print(f"\nElimination games: {df['is_elimination_game'].sum()}")

## 4. Group 1 — Team Efficiency Features

In [ ]:
eff_cols = [
    "TEAM_ID",
    "season",
    "E_OFF_RATING",
    "E_DEF_RATING",
    "E_NET_RATING",
    "E_PACE",
]
eff = metrics[eff_cols].copy()

# Home team efficiency
df = df.merge(
    eff.rename(
        columns={
            "TEAM_ID": "home_team_id",
            "E_OFF_RATING": "home_ortg",
            "E_DEF_RATING": "home_drtg",
            "E_NET_RATING": "home_net_rtg",
            "E_PACE": "home_pace",
        }
    ),
    on=["home_team_id", "season"],
    how="left",
)

# Away team efficiency
df = df.merge(
    eff.rename(
        columns={
            "TEAM_ID": "away_team_id",
            "E_OFF_RATING": "away_ortg",
            "E_DEF_RATING": "away_drtg",
            "E_NET_RATING": "away_net_rtg",
            "E_PACE": "away_pace",
        }
    ),
    on=["away_team_id", "season"],
    how="left",
)

# Differentials
df["ortg_diff"] = df["home_ortg"] - df["away_ortg"]
df["drtg_diff"] = df["home_drtg"] - df["away_drtg"]

print(
    f"Missing efficiency values: {df[['home_ortg', 'away_ortg']].isnull().sum().to_dict()}"
)

## 5. Group 2 — Team Quality Proxies

In [ ]:
# Win % and offensive rebounding % (proxies for team quality / second-chance scoring)
qual_cols = ["TEAM_ID", "season", "W_PCT", "E_OREB_PCT"]
qual = metrics[qual_cols].copy()

df = df.merge(
    qual.rename(
        columns={
            "TEAM_ID": "home_team_id",
            "W_PCT": "home_win_pct",
            "E_OREB_PCT": "home_oreb_pct",
        }
    ),
    on=["home_team_id", "season"],
    how="left",
)
df = df.merge(
    qual.rename(
        columns={
            "TEAM_ID": "away_team_id",
            "W_PCT": "away_win_pct",
            "E_OREB_PCT": "away_oreb_pct",
        }
    ),
    on=["away_team_id", "season"],
    how="left",
)

print(f"Win pct range: {df['home_win_pct'].min():.3f} – {df['home_win_pct'].max():.3f}")

## 6. Group 3 — Game Context (already computed)

In [ ]:
df["rest_advantage"] = df["home_rest_days"] - df["away_rest_days"]

print("Context features summary:")
print(
    df[
        [
            "home_rest_days",
            "away_rest_days",
            "rest_advantage",
            "game_number",
            "is_elimination_game",
        ]
    ]
    .describe()
    .round(2)
)

## 7. Group 4 — Historical Playoff Record

In [ ]:
# Build season-level playoff stats from the raw games data
# For each team-season: wins, losses, and whether they reached the Finals
playoff_history = games.copy()
playoff_history["win"] = (playoff_history["WL"] == "W").astype(int)

season_stats = (
    playoff_history.groupby(["season", "TEAM_ID"])
    .agg(
        playoff_wins=("win", "sum"),
        playoff_games=("win", "count"),
        reached_finals=("is_finals", "max"),
    )
    .reset_index()
)
season_stats["playoff_win_pct"] = (
    season_stats["playoff_wins"] / season_stats["playoff_games"]
)

# Numeric season for ordering
season_stats["season_year"] = season_stats["season"].str[:4].astype(int)


def prior_playoff_stats(
    team_id: int, current_season: str, season_stats_df: pd.DataFrame
) -> dict:
    """Return win pct over prior 3 seasons and Finals apps in prior 5 seasons."""
    current_year = int(current_season[:4])
    hist = season_stats_df[
        (season_stats_df["TEAM_ID"] == team_id)
        & (season_stats_df["season_year"] < current_year)
    ]
    prior_3 = hist[hist["season_year"] >= current_year - 3]
    prior_5 = hist[hist["season_year"] >= current_year - 5]

    win_pct = prior_3["playoff_win_pct"].mean() if len(prior_3) > 0 else np.nan
    finals_apps = int(prior_5["reached_finals"].sum()) if len(prior_5) > 0 else 0
    return win_pct, finals_apps


home_hist = df.apply(
    lambda r: prior_playoff_stats(r["home_team_id"], r["season"], season_stats), axis=1
)
away_hist = df.apply(
    lambda r: prior_playoff_stats(r["away_team_id"], r["season"], season_stats), axis=1
)

df["home_playoff_win_pct_3yr"] = [x[0] for x in home_hist]
df["home_finals_apps_5yr"] = [x[1] for x in home_hist]
df["away_playoff_win_pct_3yr"] = [x[0] for x in away_hist]
df["away_finals_apps_5yr"] = [x[1] for x in away_hist]

print(
    f"Historical win pct missing (first season expected): {df['home_playoff_win_pct_3yr'].isnull().sum()}"
)
print(
    df[
        [
            "home_playoff_win_pct_3yr",
            "away_playoff_win_pct_3yr",
            "home_finals_apps_5yr",
            "away_finals_apps_5yr",
        ]
    ]
    .describe()
    .round(3)
)

## 8. Assemble Final Feature Matrix

## 8. Group 5 — In-Series Momentum Features

In [ ]:
# home_margin: home team point differential for each game (positive = home won)
df["home_margin"] = df["PLUS_MINUS"]  # PLUS_MINUS in home row = home team's +/-

df = df.sort_values(["series_key", "GAME_DATE"]).reset_index(drop=True)

# Last game margin in the series (0 for game 1 — no prior context)
df["last_game_margin"] = df.groupby("series_key")["home_margin"].shift(1).fillna(0)

# Cumulative point differential up to (not including) this game
df["cumulative_pts_diff"] = (
    df.groupby("series_key")["home_margin"]
    .apply(lambda x: x.shift(1).cumsum().fillna(0))
    .reset_index(level=0, drop=True)
)

# Did home team win the last game? (0 for game 1)
df["home_won_last_game"] = (df["last_game_margin"] > 0).astype(int)
# Mark game 1 as 0 (neutral — no prior game)
df.loc[df["game_number"] == 1, "home_won_last_game"] = 0

print("In-series feature sample (games 2+ in a series):")
cols = [
    "series_key",
    "game_number",
    "home_series_wins",
    "away_series_wins",
    "last_game_margin",
    "cumulative_pts_diff",
    "home_won_last_game",
    "home_win",
]
print(df[df["game_number"] > 1][cols].head(8).to_string(index=False))

In [ ]:
FEATURE_COLS = [
    # Group 1 — efficiency
    "home_ortg",
    "away_ortg",
    "home_drtg",
    "away_drtg",
    "home_net_rtg",
    "away_net_rtg",
    "home_pace",
    "away_pace",
    "ortg_diff",
    "drtg_diff",
    # Group 2 — team quality proxies
    "home_win_pct",
    "away_win_pct",
    "home_oreb_pct",
    "away_oreb_pct",
    # Group 3 — game context
    "home_rest_days",
    "away_rest_days",
    "rest_advantage",
    "game_number",
    "is_elimination_game",
    # Group 4 — historical playoff record
    "home_playoff_win_pct_3yr",
    "away_playoff_win_pct_3yr",
    "home_finals_apps_5yr",
    "away_finals_apps_5yr",
    # Group 5 — in-series momentum
    "home_series_wins",
    "away_series_wins",
    "last_game_margin",
    "cumulative_pts_diff",
    "home_won_last_game",
]
META_COLS = ["GAME_ID", "season", "GAME_DATE", "home_team_id", "away_team_id", "round"]
TARGET_COL = "home_win"

out = df[META_COLS + FEATURE_COLS + [TARGET_COL]].copy()

# Fill historical win pct NaN with 0.5 (neutral prior for teams with no history)
out["home_playoff_win_pct_3yr"] = out["home_playoff_win_pct_3yr"].fillna(0.5)
out["away_playoff_win_pct_3yr"] = out["away_playoff_win_pct_3yr"].fillna(0.5)

print(f"Shape: {out.shape}")
print(f"Features: {len(FEATURE_COLS)}")
print(
    f"Missing values:\n{out[FEATURE_COLS].isnull().sum()[out[FEATURE_COLS].isnull().sum() > 0]}"
)
print(f"\nTarget distribution: {out[TARGET_COL].value_counts().to_dict()}")

## 9. Validate + Save

In [ ]:
assert len(out) >= 600, f"Expected >=600 rows, got {len(out)}"
assert set(FEATURE_COLS).issubset(out.columns), "Missing feature columns"
assert out[TARGET_COL].isnull().sum() == 0, "Target has nulls"
assert out[TARGET_COL].isin([0, 1]).all(), "Target is not binary"

out.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved {len(out)} rows to {OUTPUT_PATH}")
print("\nSample row:")
out[FEATURE_COLS].describe().round(3)